# Collect a diverse multi-modal dataset

Collect random-policy rollouts from several Gymnasium environments — spanning every
observation and action modality — into a single `DatasetStore`.

The rows we write follow the **mouse-env rollout contract** holistically (see
`MouseEnvRecord` and the contract section in docs/dataset.md). The store, the
parquet on disk, and what you push to the Hub contain the nested `observation`/
`action` structs + scalars (`reward`, `done`, `time`, `group_id`, `episode_index`,
`reward_episodic`, ...). The flat `TensorDict` shape is produced only at batch time.

| Environment | Observation | Action |
|---|---|---|
| `FrozenLake-v1`, `Taxi-v4` | discrete | discrete |
| `CartPole-v1` | continuous | discrete |
| `Pendulum-v1` | continuous | continuous |
| `ALE/Breakout-v5` | image (84×84) | discrete |

One store holds them all. The encoder (driven by sizes you pass to
`PrefetchBatchifier`) projects the contract fields to the tensors the model
pipeline expects.

> **Note:** continuous actions are stored in the dataset (as part of the contract),
> but current heads are discrete-action only.

Requires the examples + atari extras: `pip install -e ".[all]"`.

In [6]:
import numpy as np
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing

from mouse_core.data import DatasetStore, push_stores_to_hub

gym.register_envs(ale_py)  # register the ALE/* Atari environments

IMG_SIZE = 84  # Atari frames are preprocessed to IMG_SIZE x IMG_SIZE grayscale

# Environments spanning every observation/action modality. They are collected into
# one store; modalities absent from a given step are zero-filled at encode time.
ENV_SPECS = [
    {"id": "FrozenLake-v1",   "kwargs": {"is_slippery": True}, "obs": "discrete",   "action": "discrete"},
    {"id": "Taxi-v3",         "kwargs": {},                    "obs": "discrete",   "action": "discrete"},
    {"id": "CartPole-v1",     "kwargs": {},                    "obs": "continuous", "action": "discrete"},
    {"id": "Pendulum-v1",     "kwargs": {},                    "obs": "continuous", "action": "continuous"},
    {"id": "ALE/Breakout-v5", "kwargs": {"frameskip": 1},      "obs": "image",      "action": "discrete",
     "atari": True, "episodes": 3, "max_steps": 50},
]
EPISODES_PER_ENV = 10
MAX_STEPS = 100

## Collect rollouts

We append rows that directly follow the mouse-env contract (see `MouseEnvRecord`).
The data written to the `DatasetStore` (and later to parquet) *is* that contract —
nested observation/action plus the rollout scalars. The only place the shape changes
is when `PrefetchBatchifier` encodes a training batch for the model.

For each env we also set `group_id`, `episode_index`, and `reward_episodic` so the
stored dataset is a realistic example of what you get from real mouse-env collection.

When using mouse-env, you pair the action dict you passed to `step()` with the
`result` it produced. The contract (including `done` semantics 0/1/2, `reward_episodic`,
`q_star`, etc.) is the source of truth throughout mouse-core.

In [7]:
def make_env(spec):
    env = gym.make(spec["id"], **spec["kwargs"])
    if spec.get("atari"):
        env = AtariPreprocessing(env, grayscale_obs=True, scale_obs=False, screen_size=IMG_SIZE)
    return env


def encode_obs(obs, kind):
    if kind == "discrete":
        return {"discrete": int(obs)}
    if kind == "continuous":
        return {"continuous": np.asarray(obs, dtype=np.float32).ravel().tolist()}
    return {"image": np.asarray(obs).astype(np.int32).ravel().tolist()}


def encode_action(action, kind):
    if kind == "discrete":
        return {"discrete": int(action)}
    return {"continuous": np.asarray(action, dtype=np.float32).ravel().tolist()}


def collect_episodes(store, env, spec, num_episodes, max_steps, group_id):
    """Append using the mouse-env record contract (action paired with result)."""
    obs_kind, act_kind = spec["obs"], spec["action"]
    for ep in range(num_episodes):
        obs, _ = env.reset()
        # Seed: the action 'taken' to reach the first obs (convention in mouse-env data).
        action = {"discrete": 0} if act_kind == "discrete" else {"continuous": [0.0]}
        reward = 0.0
        done_flag = 0
        reward_ep = 0.0

        for step_idx in range(max_steps):
            store.append({
                "observation": encode_obs(obs, obs_kind),
                "action": action,
                "reward": reward,
                "done": done_flag,
                "time": step_idx,
                "group_id": group_id,
                "episode_index": ep,
                "reward_episodic": reward_ep,
            })

            raw_action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(raw_action)
            action = encode_action(raw_action, act_kind)
            reward = float(reward)
            done_flag = 1 if terminated else (2 if truncated else 0)
            reward_ep = float(reward)  # simplistic; real shaped reward from mouse-env

            if terminated or truncated:
                store.append({
                    "observation": encode_obs(obs, obs_kind),
                    "action": action,
                    "reward": reward,
                    "done": done_flag,
                    "time": step_idx + 1,
                    "group_id": group_id,
                    "episode_index": ep,
                    "reward_episodic": reward_ep,
                })
                break


store = DatasetStore()

for spec in ENV_SPECS:
    env = make_env(spec)
    gid = f"{spec['id']}#demo"
    collect_episodes(store, env, spec, spec.get("episodes", EPISODES_PER_ENV), spec.get("max_steps", MAX_STEPS), gid)
    env.close()
    print(f"{spec['id']}: {len(store)} total steps")

print(store)

FrozenLake-v1: 57 total steps
Taxi-v3: 1046 total steps
CartPole-v1: 1267 total steps
Pendulum-v1: 2267 total steps
ALE/Breakout-v5: 2417 total steps
DatasetStore(steps=2417)


## Optional: push to the Hugging Face Hub

Set `REPO_ID` (e.g. `your-org/my-mouse-dataset`) to upload.

In [8]:
REPO_ID = "mouse-core-test"  # e.g. "your-org/my-mouse-dataset"

if REPO_ID:
    # Use config_name for a bin; rows follow the mouse-env contract.
    # The push deletes the prior README and lets HF write a fresh card whose
    # dataset_info: matches the data just uploaded (important for new datasets).
    # After the push, edit the Hub README to add a configs: YAML block if you
    # want declarative splits/subsets per the HF repository structure guide.
    url = push_stores_to_hub(
        [store], repo_id=REPO_ID, split="train", config_name="demo_multi_modal", private=False
    )
    print(f"Pushed to {url}")
else:
    print("Set REPO_ID to upload this dataset to the Hugging Face Hub.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-core-test (train: 2417 steps)
Pushed to https://huggingface.co/datasets/micahr234/mouse-core-test
